# Frozen temporal VUS evaluation
Evaluate the verified notebook 04 bundle without fitting or selecting models on test scores. This previously examined interval is not a fresh external test. These are three-class future-reclassification results, not five-class diagnosis or patient disease probabilities. Missing-target ranking metrics remain undefined; aggregate AP requires both targets.


In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
import sys

PROJECT_ROOT = Path(r"D:\GeneVISTA")
processed_folder = PROJECT_ROOT / "data" / "processed"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation_validation import (
    resolve_bundle, load_evaluation_data, verify_development,
    verify_predictions, sha256, safe_file, fixed_balanced_accuracy,
)
from src.training_validation import ranking_metrics, safe_ratio

# Use the successfully verified run from notebook 04, not an old hardcoded model.
evaluation_bundle = resolve_bundle(PROJECT_ROOT)
evaluation_plan = json.loads((evaluation_bundle / "evaluation_plan.json").read_text(encoding="utf-8"))
if sklearn.__version__ != evaluation_plan["sklearn_version"]:
    raise RuntimeError("scikit-learn differs from the frozen training environment")
development_provenance = verify_development(PROJECT_ROOT, evaluation_bundle)
feature_config = evaluation_plan["feature_config"]
categorical_features = feature_config["categorical_features"]
numeric_features = feature_config["numeric_features"]
model_features = categorical_features + numeric_features
join_keys = feature_config["join_keys"]
X_test, y_test, test_ids, majority_class = load_evaluation_data(processed_folder, evaluation_plan)
frozen_models = {}
model_names = [evaluation_plan["primary_model"], evaluation_plan["comparator_model"]]
assert len(set(model_names)) == 2
for name in model_names:
    model = joblib.load(safe_file(evaluation_bundle, f"{name}.joblib"))
    assert list(model.feature_names_in_) == model_features
    assert set(model.classes_) == set(feature_config["target_classes"])
    frozen_models[name] = model
print("FROZEN TEST SETUP VERIFIED")
print(f"Test interval: {evaluation_plan['test_interval']}; examples: {len(X_test):,}")
print("Loaded models:", ", ".join(frozen_models))
print("This interval has been evaluated previously; it is not a fresh external test.")


FROZEN TEST SETUP VERIFIED
Test interval: 2025-01_to_2026-01; examples: 1,456,851
Loaded models: primary_temporal_gradient_boosting, comparator_temporal_balanced
This interval has been evaluated previously; it is not a fresh external test.


In [2]:
from datetime import datetime
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    average_precision_score
)

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
results_folder = evaluation_bundle / f"test_evaluation_{run_name}"
results_folder.mkdir(parents=True, exist_ok=False)

class_names = feature_config["target_classes"]
ranking_targets = ["became_benign", "became_pathogenic"]

metric_rows = []
ranking_rows = []
review_rows = []

tie_order = np.random.default_rng(
    evaluation_plan["tie_break_seed"]
).permutation(len(y_test))

# Majority reference comes from training counts, never the test labels.
majority_predictions = np.full(len(y_test), majority_class)

metric_rows.append({
    "model": "majority_class_reference",
    "accuracy": accuracy_score(y_test, majority_predictions),
    "balanced_accuracy": fixed_balanced_accuracy(
        y_test, majority_predictions
    ),
    "macro_f1": f1_score(
        y_test,
        majority_predictions,
        labels=class_names,
        average="macro",
        zero_division=0
    )
})

for name, model in frozen_models.items():
    print(f"Evaluating {name}...")

    probabilities = model.predict_proba(X_test)
    model_classes = list(model.named_steps["classifier"].classes_)

    assert probabilities.shape == (len(y_test), len(class_names))
    assert set(model_classes) == set(class_names)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
    np.testing.assert_allclose(
        probabilities.sum(axis=1), 1.0, atol=1e-6
    )

    predictions = np.asarray(model_classes)[
        probabilities.argmax(axis=1)
    ]

    metric_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": fixed_balanced_accuracy(
            y_test, predictions
        ),
        "macro_f1": f1_score(
            y_test,
            predictions,
            labels=class_names,
            average="macro",
            zero_division=0
        )
    })

    report = classification_report(
        y_test,
        predictions,
        labels=class_names,
        output_dict=True,
        zero_division=0
    )

    pd.DataFrame(report).T.to_csv(
        results_folder / f"{name}_classification_report.csv"
    )

    matrix = pd.DataFrame(
        confusion_matrix(y_test, predictions, labels=class_names),
        index=pd.Index(class_names, name="actual"),
        columns=pd.Index(class_names, name="predicted")
    )

    assert int(matrix.to_numpy().sum()) == len(y_test)
    matrix.to_csv(results_folder / f"{name}_confusion_matrix.csv")

    print(
        classification_report(
            y_test,
            predictions,
            labels=class_names,
            digits=4,
            zero_division=0
        )
    )

    for target in ranking_targets:
        actual = y_test.eq(target).to_numpy()
        total_positives = int(actual.sum())

        scores = probabilities[:, model_classes.index(target)]
        prevalence, ap, lift = ranking_metrics(actual, scores)

        ranking_rows.append({
            "model": name,
            "target": target,
            "prevalence": prevalence,
            "average_precision": ap,
            "AP_divided_by_prevalence": lift
        })

        ranked = tie_order[
            np.argsort(-scores[tie_order], kind="stable")
        ]

        for size in sorted({min(v, len(y_test)) for v in evaluation_plan["review_sizes"]}):
            selected = ranked[:size]
            correct = int(actual[selected].sum())

            review_rows.append({
                "model": name,
                "target": target,
                "review_size": size,
                "true_reclassifications": correct,
                "other_outcomes": size - correct,
                "precision_percent": 100 * safe_ratio(correct, size),
                "recall_percent": 100 * safe_ratio(correct, total_positives),
                "lift_over_prevalence": safe_ratio(safe_ratio(correct, size), prevalence)
            })

    # Save traceable scores for later auditing.
    prediction_table = test_ids.reset_index(drop=True).copy()
    prediction_table["actual_target"] = y_test.to_numpy()
    prediction_table["predicted_target"] = predictions

    for column, target in enumerate(model_classes):
        prediction_table[f"score_{target}"] = probabilities[:, column]

    prediction_table.to_csv(
        results_folder / f"{name}_test_predictions.csv.gz",
        index=False,
        compression="gzip",
        chunksize=100_000
    )

    verify_predictions(results_folder / f"{name}_test_predictions.csv.gz", prediction_table)
    del prediction_table, probabilities, predictions

test_metrics = pd.DataFrame(metric_rows)
test_ranking = pd.DataFrame(ranking_rows)
test_review_lists = pd.DataFrame(review_rows)

primary_metric = (
    test_ranking.groupby("model", as_index=False)["average_precision"]
    .agg(lambda values: values.mean() if values.notna().all() else np.nan)
    .rename(columns={
        "average_precision": "mean_reclassification_average_precision"
    })
)

for filename, table in {
    "test_metrics.csv": test_metrics,
    "test_ranking.csv": test_ranking,
    "test_review_lists.csv": test_review_lists,
    "primary_metric.csv": primary_metric
}.items():
    table.to_csv(results_folder / filename, index=False)

evaluation_record = {
    "evaluation_plan": evaluation_plan,
    "test_rows": len(y_test),
    "models_evaluated": list(frozen_models),
    "test_evaluated": True,
    "fitting_performed": False,
    "majority_reference_class": majority_class,
    "prediction_artifacts_verified": True,
    "test_is_fresh_external_validation": False,
    "scores_are_calibrated_probabilities": False,
    "class_support": {str(k): int(v) for k, v in y_test.value_counts().items()},
    "plan_sha256": sha256(evaluation_bundle / "evaluation_plan.json"),
    "model_sha256": {name: sha256(evaluation_bundle / f"{name}.joblib") for name in frozen_models},
    "test_input_sha256": {
        key: sha256(processed_folder / folder / evaluation_plan["test_files"][key])
        for key, folder in [("baseline_features", "model_features"), ("temporal_features", "temporal_features_v1"), ("labels", "model_labels")]
    },
    "completed_at": datetime.now().isoformat()
}

(results_folder / "evaluation_record.json").write_text(
    json.dumps(evaluation_record, indent=2),
    encoding="utf-8"
)

print("\nHELD-OUT EVALUATION COMPLETE")
print(f"Results saved to: {results_folder}")

print("\nCLASSIFICATION METRICS")
display(test_metrics)

print("\nPRIMARY METRIC")
display(primary_metric)

print("\nRANKING QUALITY")
display(test_ranking)

print("\nREVIEW-LIST RESULTS")
print(test_review_lists.round(4).to_string(index=False))

Evaluating primary_temporal_gradient_boosting...
                   precision    recall  f1-score   support

       stayed_vus     0.9937    1.0000    0.9968   1447648
    became_benign     0.0000    0.0000    0.0000      7098
became_pathogenic     0.0000    0.0000    0.0000      2105

         accuracy                         0.9937   1456851
        macro avg     0.3312    0.3333    0.3323   1456851
     weighted avg     0.9874    0.9937    0.9905   1456851

Evaluating comparator_temporal_balanced...
                   precision    recall  f1-score   support

       stayed_vus     0.9934    0.2921    0.4514   1447648
    became_benign     0.0044    0.6037    0.0088      7098
became_pathogenic     0.0133    0.3758    0.0257      2105

         accuracy                         0.2937   1456851
        macro avg     0.3370    0.4238    0.1620   1456851
     weighted avg     0.9872    0.2937    0.4486   1456851


HELD-OUT EVALUATION COMPLETE
Results saved to: D:\GeneVISTA\models\temporal